# Laboratorio 10 — Dos escritores al mismo tiempo

**Duración: 45 minutos.**

Hasta hoy cada uno trabajó en su propia tabla. Hoy los siete van a escribir en **la
misma**, a la vez, y a propósito.

No es un ejercicio de laboratorio: es lo que pasa todas las noches en una plataforma de
datos. Dos procesos de carga que arrancan a la misma hora y tocan la misma tabla. La
pregunta del día es qué ocurre entonces, y la respuesta tiene dos partes que conviene no
confundir.

---

### En el almacén

Dos personas quieren anotar en la misma libreta al mismo tiempo.

- Si cada una anota **una cosa nueva** al final, no hay problema: las dos anotaciones
  caben y quedan las dos.
- Si las dos quieren **corregir el mismo renglón**, hay problema: alguna de las dos va a
  escribir encima de la otra, y eso no puede pasar sin que nadie se entere.

Iceberg trata esos dos casos de forma distinta, y hoy vas a ver los dos.

La tabla de hoy se llama `curso.escritores_lab10` y es **una sola para toda la sala**. No está en tu
espacio: está en el espacio compartido.


## Paso 0 — La tabla compartida

Primero mira lo que hay. No hace falta `USE`: la tabla se nombra completa, con su
database adelante, porque no es tuya.

In [1]:
%%sql
-- Celda 0.1
SELECT * FROM curso.escritores_lab10 ORDER BY rut

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/24 20:49:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


rut,razon_social,segmento
76171162-8,Huemul Alimentos EIRL,GRANDE
77746521-K,Pehuen Logistica EIRL,MICRO
77884562-8,Araucaria Ferreteria SpA,PEQUENA
78800840-6,Copihue Maquinarias EIRL,MEDIANA


Cuatro contribuyentes, los mismos que viste en otros laboratorios. Y una sola página
en la libreta, la de la carga inicial.

In [2]:
%%sql
-- Celda 0.2
SELECT snapshot_id, committed_at, operation
FROM curso.escritores_lab10.snapshots
ORDER BY committed_at

snapshot_id,committed_at,operation
3799075519760004269,2026-09-24 20:46:41.963,append


## Paso 1 — Dos escritores insertan a la vez

Ahora el primer experimento. Para que haya **dos escritores de verdad** hace falta que
dos cosas escriban al mismo tiempo, y tú eres una sola persona. Esta celda resuelve eso:
lanza **dos hilos**, que son dos líneas de trabajo que corren en paralelo dentro de tu
cuaderno, y cada uno inserta su fila.

Ya viene escrita. Léela antes de correrla: `threading.Thread` crea cada hilo, `start()`
los echa a andar y `join()` espera a que los dos terminen.

Los dos van con el **mismo RUT**, y es a propósito: la tabla va a quedar con dos filas del
mismo contribuyente. Hoy eso no importa, porque lo que estamos mirando es qué pasa cuando
dos escriben a la vez, no la llave del padrón.

> **Si prefieres verlo a mano**, se puede: abre este mismo cuaderno en **dos pestañas**
> del navegador, escribe el `INSERT` en las dos y ejecútalas una tras otra lo más rápido
> que puedas. Es lo que se hace en clase, con una persona por pestaña. Los hilos son la
> versión que no depende de tus dedos.

In [3]:
# Celda 1.1
import threading

def escribe(nombre):
    spark.sql(
        f"INSERT INTO curso.escritores_lab10 VALUES "
        f"('99000001-1', 'Escritor {nombre} SpA', 'MICRO')"
    )
    print(nombre, "escribio")

hilos = [threading.Thread(target=escribe, args=(n,)) for n in ("uno", "dos")]
for h in hilos:
    h.start()
for h in hilos:
    h.join()

print("los dos terminaron")

dos escribio
uno escribio
los dos terminaron


A ninguno de los dos le falló. Cuenta las filas.

In [4]:
%%sql
-- Celda 1.2
SELECT count(*) AS filas FROM curso.escritores_lab10

filas
6


Están las seis: las cuatro iniciales más una por cada hilo. Ahora mira la libreta.

In [5]:
%%sql
-- Celda 1.3
SELECT snapshot_id, committed_at, operation
FROM curso.escritores_lab10.snapshots
ORDER BY committed_at

snapshot_id,committed_at,operation
3799075519760004269,2026-09-24 20:46:41.963,append
3134416562057602279,2026-09-24 20:50:00.930,append
4844310876365237887,2026-09-24 20:50:01.203,append


## Paso 2 — Todos corrigen la misma fila

Una página nueva por cada hilo, las dos `append`, en el orden en que llegaron.
**Ninguno perdió su fila y ninguno tuvo que esperar al otro.**

Ahora el segundo experimento, y es el interesante. Esta vez los dos no agregan algo
nuevo: **corrigen la misma fila**, al mismo tiempo, y cada uno le pone un segmento
distinto.

In [6]:
# Celda 2.1
import threading

def corrige(nombre, segmento):
    try:
        spark.sql(
            f"UPDATE curso.escritores_lab10 SET segmento = '{segmento}' "
            f"WHERE rut = '77746521-K'"
        )
        print(nombre, "corrigio a", segmento)
    except Exception as error:
        print(nombre, "NO pudo:", str(error).splitlines()[0][:90])

hilos = [
    threading.Thread(target=corrige, args=("uno", "GRANDE")),
    threading.Thread(target=corrige, args=("dos", "PEQUENA")),
]
for h in hilos:
    h.start()
for h in hilos:
    h.join()

print("los dos terminaron")

dos corrigio a PEQUENA
26/09/24 20:50:03 ERROR ReplaceDataExec: Data source write support IcebergBatchWrite(table=spark_catalog.curso.escritores_lab10, format=PARQUET) is aborting.
26/09/24 20:50:03 ERROR ReplaceDataExec: Data source write support IcebergBatchWrite(table=spark_catalog.curso.escritores_lab10, format=PARQUET) aborted.


uno NO pudo: An error occurred while calling o39.sql.
los dos terminaron


**A varios les falló.** Si eres uno de ellos, busca en el error la línea que empieza
con `Caused by` y vas a encontrar esto:

```
org.apache.iceberg.exceptions.ValidationException: Found conflicting files that can
contain records matching ref(name="rut") == "77746521-K"
```

No es un error tuyo. Es Iceberg diciendo que alguien más cambió esa misma fila mientras
tú preparabas tu cambio, y que por eso **no va a escribir encima**.

Mira cómo quedó.

In [7]:
%%sql
-- Celda 2.2
SELECT * FROM curso.escritores_lab10 WHERE rut = '77746521-K'

rut,razon_social,segmento
77746521-K,Pehuen Logistica EIRL,PEQUENA


Uno solo ganó, y su nombre está ahí. Ahora la historia completa.

In [8]:
%%sql
-- Celda 2.3
SELECT made_current_at, snapshot_id, is_current_ancestor
FROM curso.escritores_lab10.history
ORDER BY made_current_at

made_current_at,snapshot_id,is_current_ancestor
2026-09-24 20:46:41.963,3799075519760004269,True
2026-09-24 20:50:00.930,3134416562057602279,True
2026-09-24 20:50:01.203,4844310876365237887,True
2026-09-24 20:50:03.243,8167145358722409803,True


## Paso 3 — Concurrencia optimista

Lo que acabas de ver tiene nombre: **concurrencia optimista**.

**Iceberg no bloquea.** No existe un candado que un proceso ponga sobre la tabla para que
los demás esperen. Cada escritor prepara su cambio por su cuenta, sin pedir permiso, y
solo al final intenta anotarlo en la libreta.

Ahí pasa una de tres cosas:

- **Nadie tocó nada mientras tanto.** Se anota y listo.
- **Alguien anotó algo que no se pisa con lo tuyo**, como en el paso 1, donde cada uno
  agregó una fila distinta. Iceberg **reintenta solo**, contra la página nueva, y lo
  consigue. Por eso el paso 1 no falló y nadie vio nada.
- **Alguien tocó los mismos archivos que tú**, como en el paso 2. Ahí no hay reintento que
  valga: reintentar sería escribir encima de una corrección ajena. Iceberg **falla a
  propósito**.

Esa tercera es la que importa en producción, y es la que casi nadie espera la primera vez.

---

### Qué hace un proceso de carga bien escrito

Si el commit le rebota:

1. **No se rinde ni sigue como si nada.** Un error de commit no es un aviso.
2. **Vuelve a leer la tabla** y rehace su cálculo sobre lo que hay ahora, no sobre lo que
   había cuando empezó.
3. **Reintenta unas pocas veces**, con una espera que crece entre intento e intento.
4. **Si sigue rebotando, avisa.** Que dos procesos peleen siempre por las mismas filas es
   un problema de diseño, no de suerte, y hay que arreglarlo arriba: separando las cargas
   por partición, por ejemplo, para que no se toquen.

Lo que **no** hay que hacer es el reintento ciego sin releer. Eso reintroduce justo lo que
Iceberg estaba evitando.

## Cierre

Dos cosas distintas, y conviene no confundirlas nunca más:

| Lo que pasó | Qué hizo Iceberg | Qué viste |
|---|---|---|
| Cada uno agregó una fila distinta | reintentó solo | nada, funcionó |
| Todos corrigieron la misma fila | falló a propósito | `ValidationException` |

**Ninguna de las dos perdió un dato.** Ni en la primera, donde las siete filas están, ni
en la segunda, donde el que ganó dejó su corrección y los demás se enteraron de que no
pudieron.

Eso es lo que hace que una tabla Iceberg aguante que varios procesos escriban sin
coordinarse. No hay un coordinador: hay una regla, y la regla es que nadie escribe encima
de lo que no leyó.
